# Reading and Writing Binary Records

CSC-239 · Module 11 · Lesson 1 of 2

You can read and write text files, close resources, and handle exceptions. Now you will store primitive values in a binary format whose reader must follow the same field agreement as its writer.

Select the **Java** kernel in your Workspace. Start with a fresh kernel and run cells in order. This notebook creates its own starting state.


## Learning Goals

- Write and read a primitive binary record using a matching field schema.
- Detect an incomplete record and explain why matching byte lengths alone do not prove matching field meaning.


## Why This Matters

A room record may contain a room number, an available-seat count, and an open flag. Another part of the program needs to restore those exact kinds of values without guessing their meaning from raw bytes.


## Check Your Starting Point

Recall why a Path identifies a location rather than opening a file. Explain what try-with-resources closes and why finally can perform cleanup after an error. Contrast a collection-processing stream from Module 10 with an I/O stream.

**My explanation:**


## Concept

### Read and write bytes

A **byte stream** reads or writes a sequence of bytes. A byte is a small unit of stored data. A binary file is still a file of bytes, but its contents are not necessarily an ordinary text document. The reader needs the format that gives those bytes meaning.

Files.newOutputStream opens raw byte output to a path. Files.newInputStream opens raw byte input. A DataOutputStream wraps the output to write primitive Java values, and a DataInputStream wraps the input to read them back.

You already used wrapper objects with buffered character streams. Here the wrapper supplies typed binary operations instead of line-oriented text operations. A .bin filename is a helpful label for people; its extension does not enforce the file's format.

### Write primitive values with matching operations

**Typed binary output** writes values through operations such as writeInt, writeDouble, and writeBoolean. The operation determines how its value is represented.

**Typed binary input** reads values through corresponding operations such as readInt, readDouble, and readBoolean. The reader consumes the next bytes from its current position. It does not search for a variable name inside the file.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("count-example-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(6);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        System.out.println("Count: " + reader.readInt());
    }
} finally {
    Files.deleteIfExists(file);
}
```

This prints Count: 6. The writer closes before the reader opens. Closing each data stream also closes its underlying stream. The final cleanup deletes this example's private temporary file.

The checked I/O operations can fail. IJava permits these top-level demonstration calls; in a conventional Java method, handle or declare the checked exception as you practiced in Module 7. The video uses a main method with a throws declaration.

### Agree on field types and order

A **binary record schema** is the agreed field types and order used by both writer and reader. Record here means one agreed group of values.

For a seat offer, one possible schema is:

| Position | Meaning | Write operation | Read operation |
| --- | --- | --- | --- |
| First | Available seats | writeInt | readInt |
| Second | Price per seat | writeDouble | readDouble |
| Third | Open for booking | writeBoolean | readBoolean |

The local variable names help a programmer understand the code. They are not field labels automatically stored by these primitive-writing operations. The agreement has to be shared by the code at both ends.

Even two fields of the same type have distinct meanings. Suppose the writer stores room number 12 and available count 3 as two int values. A reader that assigns the first int to available and the second to room gets plausible numbers with the wrong meanings.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("wrong-order-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(12);
        writer.writeInt(3);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int available = reader.readInt();
        int room = reader.readInt();
        System.out.println("Room: " + room);
        System.out.println("Available: " + available);
    }
} finally {
    Files.deleteIfExists(file);
}
```

This complete diagnostic prints Room: 3 and Available: 12. It runs successfully, but it is intentionally wrong about the meanings. A successful read is not proof that the schema was followed. Repair the assignment order when writing the actual reader.

These data-stream encodings provide portable primitive representations. They do not add a field-name dictionary or check your application's interpretation. The supplied project interface and shared classes will define a different agreement for the final project; do not replace it with this tutorial schema.

### Recognize an incomplete record

An **incomplete binary record** ends before all required fields or bytes have been read. A typed read such as readInt needs all bytes for its value. If the stream ends too early, it throws EOFException, an IOException subtype. EOF means end of file.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.io.EOFException;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("incomplete-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(12);
        writer.writeInt(3);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int room = reader.readInt();
        int available = reader.readInt();
        boolean open = reader.readBoolean();
        System.out.println("Complete record.");
    } catch (EOFException exception) {
        System.out.println("Incomplete room record.");
    }
} finally {
    Files.deleteIfExists(file);
}
```

This prints Incomplete room record. The file has the two integer fields but lacks the required boolean. The program does not invent false as a substitute. No complete-record message appears because the required read failed first.

The meaning of reaching the end depends on the schema. After one complete record, no more fields are required in these examples. While a required field remains, reaching the end is a failure. Do not catch every IOException and quietly return a default record.

### Test values, meaning, and cleanup

Use a newly created temporary file for each complete test. Predict the restored values, write them, close the writer, and read them in the documented order. Compare each named field rather than checking only that the program reached its last line.

Include zero values and both boolean states. Then make a controlled incomplete file and verify the failure response. A wrong-order diagnostic and an incomplete-file diagnostic test different problems, even though both involve an incorrect record.

The examples delete only their own temporary files. They do not modify a shared project file or depend on leftover data from an earlier notebook run.


## Video Demonstration

Watch the writer and reader follow the same int, double, boolean agreement. Predict the named restored values before the program prints.

<video controls preload="metadata" width="960">
  <source src="media/01_reading_and_writing_binary_records/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/01_reading_and_writing_binary_records/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the reading and writing binary records demonstration transcript](media/01_reading_and_writing_binary_records/transcript.md).


## Worked Example

**Subgoal 1: write the agreed fields.** Store seats, price, and open status in the documented order.

**Subgoal 2: restore the same types.** Read int, double, and boolean in that same order.

**Subgoal 3: finish resource use.** Close each stream before the next phase and delete the temporary file.


In [ ]:
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(4);
        writer.writeDouble(2.5);
        writer.writeBoolean(true);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int seats = reader.readInt();
        double price = reader.readDouble();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}


Expected output:

```text
Seats: 4
Price: 2.5
Open: true
```

The reader restores four seats, a price of 2.5, and true open status. Each read matches its corresponding write. The temporary file is removed after the stream resources have closed.


## Predict, Run, Trace, and Explain

### Predict the restored fields

Before running, predict every printed line. Pair each read with the write that supplies its value. Explain which part of the program gives each restored value its meaning and why the reader starts only after the writer closes.

My predicted lines:

Matching write and read for each field:

Where each field gets its meaning:

Why the writer closes first:


In [ ]:
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(9);
        writer.writeDouble(1.75);
        writer.writeBoolean(false);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int seats = reader.readInt();
        double price = reader.readDouble();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}


Run the complete prediction program in the Workspace. Keep your original prediction and record all actual lines. Explain any correction by tracing the matching write and read. Identify where both streams close and where the temporary file is deleted. Run the whole program again and explain why that replay has its own file.

My original prediction:

Actual output:

What I confirmed or corrected:

Where the streams close and the file is deleted:

Why a full replay does not depend on the earlier file:

### Trace types, order and resource use

Complete one row for each field: position, meaning, write operation and value, and matching read operation. Explain the jobs of Files.newOutputStream, DataOutputStream, Files.newInputStream and DataInputStream. Then trace the order in which the writer closes, the reader opens and closes, and finally deletes the file. Would renaming a reader variable change the bytes already stored?

| Position | Meaning | Write and value | Matching read |
| --- | --- | --- | --- |
| First | | | |
| Second | | | |
| Third | | | |


Raw stream and data-wrapper jobs:

Resource closing and cleanup order:

Effect of renaming a reader variable:

<details>
<summary>Show answer</summary>

The writer stores an int with value 9, a double with value 1.75 and a boolean with value false, in that order. The reader uses readInt, readDouble and readBoolean in the same order and assigns each restored value to its intended meaning. It prints Seats: 9, Price: 1.75 and Open: false. A read consumes the next encoded value; it does not search for the variable name. The writer closes before the reader opens. The reader closes before finally deletes this program’s own temporary file. Replaying the whole program creates a new temporary file. Files opens raw byte access. The data-stream wrappers supply operations for primitive values. Renaming a variable does not rewrite the file or move the reader’s position; the assignment and later use determine the program’s interpretation.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(9);
        writer.writeDouble(1.75);
        writer.writeBoolean(false);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int seats = reader.readInt();
        double price = reader.readDouble();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Seats: 9
Price: 1.75
Open: false
```

Common error: Reading the fields in the order of their print statements without checking the actual read order. Assuming a variable name is stored as a field label. Treating a .bin extension as a check of the stored format.

</details>


### Distinguish an incomplete record from a stored false flag

This complete program deliberately leaves out the final boolean write and catches EOFException, the end-of-file exception explained above. Before running, predict which read cannot finish and whether any named field lines will print. Run it, retain your prediction and explain the actual output. Then create an empty-file comparison by removing the remaining two writes while retaining the empty writer block, all reads, the catch and finally cleanup. Predict and run again. Identify the first failing read in each case. Explain why an absent flag cannot be treated as a stored false value.

My prediction and first failing read for the missing flag:

Actual output and my explanation:

My empty-file prediction and first failing read:

Actual empty-file output and my explanation:

Why missing and false are different:

How resources and the temporary file are cleaned up:


In [ ]:
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.io.EOFException;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(9);
        writer.writeDouble(1.75);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int seats = reader.readInt();
        double price = reader.readDouble();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    } catch (EOFException problem) {
        System.out.println("Incomplete seat record.");
    }
} finally {
    Files.deleteIfExists(file);
}


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

The two stored values can be read, but the required boolean has not been written. readBoolean throws EOFException. All named print statements follow all three reads, so none executes. The catch prints Incomplete seat record. This is a handled incomplete-record failure, not a successful complete record or a restored false flag. The reader closes as control leaves its resource block, and finally deletes the temporary file. In the empty-file comparison, readInt is the first failing read. The same catch message therefore describes a different point of failure; use the written schema and the source to locate it.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.io.EOFException;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(9);
        writer.writeDouble(1.75);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int seats = reader.readInt();
        double price = reader.readDouble();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    } catch (EOFException problem) {
        System.out.println("Incomplete seat record.");
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Incomplete seat record.
```

Common error: Substituting false for a flag that was never read. Claiming the earlier field print statements run even though they appear after the failed read. Using an identical catch message as proof that the same read failed.

**Check case 2.** No first int is available, so readInt fails before any later read. The catch message matches the missing-flag case even though the first missing field differs. The resource blocks still close and finally removes the private file.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.io.EOFException;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int seats = reader.readInt();
        double price = reader.readDouble();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    } catch (EOFException problem) {
        System.out.println("Incomplete seat record.");
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Incomplete seat record.
```

</details>


## Guided Practice

Complete these tasks in order. The intentionally empty code cells are safe to run, but remain unfinished until you write and check your code.


### Complete the typed wrappers and price operations

Replace OUTPUT_WRAPPER, INPUT_WRAPPER, WRITE_PRICE and READ_PRICE. Choose DataOutputStream or DataInputStream for the matching wrapper, and writeDouble or readDouble for the price operation. Copy the entire completed program into the work cell, predict its output, run it and explain why each operation matches its direction and field type.

This sample is for repair:

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new OUTPUT_WRAPPER(Files.newOutputStream(file))) {
        writer.writeInt(9);
        writer.WRITE_PRICE(1.75);
        writer.writeBoolean(false);
    }
    try (DataInputStream reader = new INPUT_WRAPPER(Files.newInputStream(file))) {
        int seats = reader.readInt();
        double price = reader.READ_PRICE();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```


My four replacements and reasons:

My predicted output:

Actual output and post-run explanation:

<details>
<summary>Show answer</summary>

Use DataOutputStream for OUTPUT_WRAPPER, DataInputStream for INPUT_WRAPPER, writeDouble for WRITE_PRICE and readDouble for READ_PRICE. Files opens the raw byte stream; the matching wrapper supplies typed operations. The price is written and read as double. The writer stores an int with value 9, a double with value 1.75 and a boolean with value false, in that order. The reader uses readInt, readDouble and readBoolean in the same order and assigns each restored value to its intended meaning. It prints Seats: 9, Price: 1.75 and Open: false. A read consumes the next encoded value; it does not search for the variable name. The writer closes before the reader opens. The reader closes before finally deletes this program’s own temporary file. Replaying the whole program creates a new temporary file.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(9);
        writer.writeDouble(1.75);
        writer.writeBoolean(false);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int seats = reader.readInt();
        double price = reader.readDouble();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Seats: 9
Price: 1.75
Open: false
```

Common error: Choosing the input wrapper for writing or the output wrapper for reading. Using readInt for the price field. Leaving a placeholder in the executable program.

</details>


### Change both sides of the field agreement

Run the unchanged starter first. Change the writer so price is written before seats, followed by open status. Change the reader’s first two declarations to match that new order. Keep the print labels, print order and stored values unchanged. Predict and run the complete modified program. Explain why its printed meanings should remain the same even though its field order changed. Then use the revised schema to store zero seats, a zero price and true open status. Predict, run and explain that result too.


In [ ]:
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(9);
        writer.writeDouble(1.75);
        writer.writeBoolean(false);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int seats = reader.readInt();
        double price = reader.readDouble();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}


Unchanged starter output:

My revised field agreement:

Predicted and actual output after both order changes:

Why the named values retain their meanings:

Zero-value/true-status prediction, actual result and explanation:

<details>
<summary>Show answer</summary>

The revised agreement is double, int, boolean: price first, seats second and open status third. Both the write order and the read order change together. The print statements still use the correctly assigned seats, price and open variables, so the named output remains Seats: 9, Price: 1.75 and Open: false. Changing only the print order would not repair a disagreement between writer and reader. The additional complete test restores zero seats, price 0.0 and true; these are actual stored values rather than replacements for missing fields.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeDouble(1.75);
        writer.writeInt(9);
        writer.writeBoolean(false);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        double price = reader.readDouble();
        int seats = reader.readInt();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Seats: 9
Price: 1.75
Open: false
```

Common error: Changing the writer order while leaving the reader unchanged. Moving only print statements instead of matching the read sequence. Confusing a stored zero or true value with absence of a field.

**Additional test: `Revised double,int,boolean agreement with seats 0, price 0.0, open true`.** The same revised sequence restores explicit zero numeric values and true status. Every required field exists. The reader does not invent values when input is absent.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeDouble(0.0);
        writer.writeInt(0);
        writer.writeBoolean(true);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        double price = reader.readDouble();
        int seats = reader.readInt();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Seats: 0
Price: 0.0
Open: true
```

</details>


### Repair two fields with swapped meanings

The displayed program writes room number 25, available count 2 and open true. Its reader is intentionally wrong about the first two field meanings. Without running the draft, predict its named output and explain why it need not throw EOFException. Repair only the first two reader declarations so the first int belongs to room and the second belongs to available. Copy the complete repaired program into the work cell, run it and explain the restored meanings. Why would unchanged file length fail to detect this error?

This sample is for repair:

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("room-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(25);
        writer.writeInt(2);
        writer.writeBoolean(true);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int available = reader.readInt();
        int room = reader.readInt();
        boolean open = reader.readBoolean();
        System.out.println("Room: " + room);
        System.out.println("Available: " + available);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```


My predicted faulty output:

Why the reads can complete despite wrong meanings:

My repaired declarations:

Actual repaired output and explanation:

Why equal file length does not prove correct interpretation:

<details>
<summary>Show answer</summary>

The faulty program completes its reads and prints Room: 2, Available: 25 and Open: true. Each readInt consumes one complete int, but the first is assigned to available and the second to room. Both numeric reads use the same type and encoded size, so enough bytes exist and EOFException is not expected for this file. Its byte length also stays unchanged. The repair reads room first and available second, matching the writer’s meanings as well as its types. It prints Room: 25, Available: 2 and Open: true. This is a meaning error that normal completion cannot detect; compare each named field with the intended schema.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("room-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(25);
        writer.writeInt(2);
        writer.writeBoolean(true);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int room = reader.readInt();
        int available = reader.readInt();
        boolean open = reader.readBoolean();
        System.out.println("Room: " + room);
        System.out.println("Available: " + available);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Room: 25
Available: 2
Open: true
```

Common error: Changing the writer to conceal a reader that assigns meanings incorrectly. Assuming that two int fields are interchangeable because their types match. Treating normal completion or an unchanged byte length as proof of the schema.

</details>


## Independent Practice

### Build a room-availability record

Create a private temporary binary file containing room number 12, available count 3 and open false. Write int, int, boolean in that order and read the same types and order. Print `Room: 12`, `Available: 3` and `Open: false` on separate lines. Include every required import and all file setup in your program. Close both streams and delete the temporary file. Before running, state the field agreement and predict the output. Run and explain how the reader restores each named field. Keep the tests for the next stage separate so each complete test creates its own file.

My field agreement:

My predicted output:

My complete Java program is in the work cell.

Actual output and post-run explanation:

Where both streams close and the file is deleted:


### Test stored values and a missing required flag

Keep your baseline program. Run each test as its own complete program with fresh temporary-file creation and cleanup: first store room 12, available 0 and open false; then room 12, available 3 and open true. Predict every output before each run. For the missing-flag test, restore the baseline numeric writes but omit the boolean write. Retain all three required reads and place the named field prints after them. Import EOFException and catch it immediately after the reader’s try-with-resources block, printing `Incomplete room record.`; retain the outer finally cleanup. Predict which read fails and whether any named record lines print. Run each test and retain your own explanation. Finally, explain why swapping the two int meanings could produce plausible wrong values without that exception, and why matching file length alone cannot verify the format.

| Test | Prediction | Actual output | My explanation |
| --- | --- | --- | --- |
| Baseline: 12, 3, false | | | |
| Zero available: 12, 0, false | | | |
| True status: 12, 3, true | | | |
| Final flag omitted | | | |


First failing read in the missing-flag case:

Why missing is different from false:

Why same-type swaps and equal byte lengths can hide wrong meanings:

Resource closing and cleanup in each full test:


<details>
<summary>Show answer</summary>

The writer stores room number 12, available count 3 and false in the required int, int, boolean order. The reader restores those fields in the same order and prints Room: 12, Available: 3 and Open: false. The writer closes before reading, and the reader closes before finally deletes the private temporary file. Both int reads have the same operation and encoded size, but their positions still have different meanings. Swapping their target meanings could complete without EOFException while labeling the values incorrectly. The zero-available and true-status tests each contain a complete record and restore the exact supplied value. The missing-flag test contains both ints but lacks the required boolean. readBoolean throws EOFException before any named print executes; the handler reports Incomplete room record. Neither false nor a default record is substituted. Zero, false and true are valid stored values; an absent required field is a different condition. A meaning check compares the named values with their intended positions, while the incomplete-input test checks whether the required data is present.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("room-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(12);
        writer.writeInt(3);
        writer.writeBoolean(false);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int room = reader.readInt();
        int available = reader.readInt();
        boolean open = reader.readBoolean();
        System.out.println("Room: " + room);
        System.out.println("Available: " + available);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Room: 12
Available: 3
Open: false
```

Common error: Reading available before room even though room was written first. Reusing an already deleted temporary path without recreating the complete fixture. Omitting imports or setup and depending on an earlier notebook.

**Additional test: Room 12, zero available seats, false status.** The zero count was written as an int and restored from a complete record. It is a valid stored value, not a missing-field substitute.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("room-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(12);
        writer.writeInt(0);
        writer.writeBoolean(false);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int room = reader.readInt();
        int available = reader.readInt();
        boolean open = reader.readBoolean();
        System.out.println("Room: " + room);
        System.out.println("Available: " + available);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Room: 12
Available: 0
Open: false
```

**Additional test: Room 12, available 3, true status.** Only the stored boolean changes from the baseline; the reader restores true through the same required readBoolean call.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("room-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(12);
        writer.writeInt(3);
        writer.writeBoolean(true);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int room = reader.readInt();
        int available = reader.readInt();
        boolean open = reader.readBoolean();
        System.out.println("Room: " + room);
        System.out.println("Available: " + available);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Room: 12
Available: 3
Open: true
```

**Additional test: Baseline numeric fields written; final flag omitted and EOFException caught.** Both ints are present, but readBoolean cannot finish. No named record lines print because all three prints follow that read. The catch reports an incomplete room record, the reader closes and finally deletes the private file.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.io.EOFException;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("room-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(12);
        writer.writeInt(3);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int room = reader.readInt();
        int available = reader.readInt();
        boolean open = reader.readBoolean();
        System.out.println("Room: " + room);
        System.out.println("Available: " + available);
        System.out.println("Open: " + open);
    } catch (EOFException problem) {
        System.out.println("Incomplete room record.");
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Incomplete room record.
```

</details>


## Summary

Byte streams provide raw byte access. Data streams add operations for primitive values, but writer and reader still need the same field schema. Swapping equal-width fields can silently change meanings. An EOFException during a required field signals an incomplete record. Close resources and clean up each controlled test file.

Close the answers and explain how you would distinguish a successful round trip, a meaning mismatch, and an incomplete record.


## Reflection

Describe three fields in a record from your field. State their types and order, and explain a mistake that could produce plausible but incorrectly labeled values. Choose a boundary and a missing-field test.

**My design and explanation:**

Next, you will write and restore the state of a supported Java object rather than manually writing each primitive field.


## Supplemental Reading

- [DataOutputStream](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/DataOutputStream.html) defines primitive binary-writing operations.
- [DataInputStream](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/DataInputStream.html) defines corresponding reads and their failure behavior.
- [EOFException](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/EOFException.html) describes premature end-of-input failures.
